# 08a | Logistic regression and regularization: spoken review

Logistic regression is the baseline model for classification and the bridge from statistics to machine learning. About 40 minutes.

> **How to use this page:** Answer each question **out loud** first (or write it in your notes for this page), then open the reference answer and compare. Statistics is your home turf, so the point here is not whether you understand it but whether you can explain it clearly in 30–60 seconds. Each reference answer ends with a **Short version**: the spoken answer in a few sentences. Practice it until it comes out naturally.

## 1. The model

$$P(Y = 1 \mid X) = \sigma(X\beta), \qquad \sigma(z) = \frac{1}{1 + e^{-z}}$$

Equivalently, the log-odds are linear:

$$\log\frac{p}{1 - p} = X\beta$$

**🗣 Say it out loud:** In a logistic regression a coefficient is $\beta_j = 0.7$. How do you interpret it?

<details>
<summary><b>Reference answer</b></summary>

Holding the other variables constant, each 1-unit increase in $x_j$ raises the **log-odds** by 0.7, i.e. **multiplies the odds** by $e^{0.7} \approx 2.0$.

It is **not** "the probability increases by 0.7". The change in probability depends on the starting point: it is largest near p = 0.5 (about β/4) and small near p = 0 or 1. To report an effect on probability you need to fix values for the other variables, or compute an average marginal effect.

**Short version:** Holding the other variables fixed, a one-unit increase multiplies the odds by e to the 0.7, about 2. It's not a 0.7 change in probability; the effect on probability depends on the baseline.

</details>

**🗣 Say it out loud:** Why not just fit a 0/1 outcome with linear regression?

<details>
<summary><b>Reference answer</b></summary>

(1) Predictions can fall below 0 or above 1 and can't be used as probabilities; (2) the variance of a 0/1 outcome is p(1−p), which changes with X, so it is inherently heteroskedastic; (3) the relationship between probability and X is usually S-shaped rather than a straight line. Logistic regression solves these with a Bernoulli likelihood and a logit link.

Worth adding: the linear probability model (LPM) is still used in practice, especially in causal analysis, because its coefficients are directly differences in probability and easy to interpret; pair it with robust standard errors. Knowing this shows you're not just reciting a textbook.

**Short version:** Linear regression can predict outside zero and one, the error variance depends on X, and the relationship is rarely linear in probability. Logistic regression uses a Bernoulli likelihood with a logit link. That said, the linear probability model is still used for easy interpretation with robust standard errors.

</details>

**🗣 Say it out loud:** How are the parameters of a logistic regression estimated? What is the loss function?

<details>
<summary><b>Reference answer</b></summary>

Maximum likelihood. The negative log-likelihood is the **cross-entropy / log loss**:

$$-\frac{1}{n}\sum_i \big[y_i \log \hat p_i + (1 - y_i)\log(1 - \hat p_i)\big]$$

It has no closed-form solution, but it is convex, so it is solved iteratively (Newton / IRLS, gradient descent, L-BFGS). The gradient has a neat form: $\frac{1}{n} X^\top(\hat p - y)$.

If the data are perfectly linearly separable, the MLE does not exist (the coefficients go to infinity), and you need regularization.

**Short version:** By maximum likelihood, which is equivalent to minimizing log loss, or cross-entropy. There's no closed form but the objective is convex, so we use Newton's method or gradient-based optimizers. With perfectly separable data the MLE diverges, which is one reason to regularize.

</details>

## 2. Regularization

Add a penalty on coefficient size to the loss function:

- **Ridge / L2:** $\lambda \sum \beta_j^2$. Shrinks coefficients toward 0, but generally not exactly to 0;
- **Lasso / L1:** $\lambda \sum |\beta_j|$. Can push some coefficients to **exactly 0**, which amounts to automatic variable selection;
- **Elastic Net:** a mix of the two.

The larger λ, the stronger the penalty and the simpler the model: bias goes up, variance goes down. λ is chosen by cross-validation.

**🗣 Say it out loud:** What is the difference between L1 and L2 regularization? Why does L1 produce sparse solutions?

<details>
<summary><b>Reference answer</b></summary>

L2 penalizes squared coefficients and shrinks all of them proportionally; correlated variables share the weight, and the result is stable but not sparse. L1 penalizes absolute values and pushes unimportant coefficients to exactly 0.

**Why sparse:** geometrically, the L1 constraint region is a diamond with corners, and the loss contours tend to touch it at a corner on an axis, which corresponds to some coefficients being 0; the L2 constraint region is a circle with no corners. Another way to say it: the L1 penalty's gradient has constant magnitude (it pushes toward 0 with the same force no matter how small the coefficient), while the L2 gradient goes to 0 as the coefficient shrinks, so it never pushes all the way to 0.

**How to choose:** use L1 when you believe only a few variables matter or you need interpretable variable selection; use L2 when there are many correlated variables that each contribute a little; when unsure, use Elastic Net.

**Short version:** L2 shrinks all coefficients smoothly and spreads weight across correlated features; L1 can set coefficients exactly to zero, giving feature selection. Geometrically the L1 constraint region has corners on the axes where the solution tends to land. I'd use L1 when I expect few relevant features and L2 when many correlated features each matter a little.

</details>

**🗣 Say it out loud:** Why do you usually standardize features before regularizing?

<details>
<summary><b>Reference answer</b></summary>

The penalty acts directly on coefficient size, and coefficient size depends on the feature's units. The same variable measured in meters vs. millimeters has coefficients 1000 times apart and gets penalized completely differently. After standardizing, all features are on the same scale and the penalty is fair. Other things that need standardization: distance-based methods (kNN, k-means, SVM), PCA, and models trained by gradient descent. Tree models don't.

Note: the standardization parameters must be computed on the training set only (the data leakage covered in 02e).

**Short version:** The penalty acts on coefficient magnitudes, which depend on feature units, so without scaling the penalty is arbitrary across features. Scaling parameters must be fit on the training data only.

</details>

## 3. Class imbalance

When positives are only 1% of the data, "predict everything negative" already has 99% accuracy. Accuracy is meaningless here.

**🗣 Say it out loud:** What are precision and recall? When do you care more about each?

<details>
<summary><b>Reference answer</b></summary>

**Precision** = TP / (TP + FP): of those predicted positive, how many really are. **Recall** = TP / (TP + FN): of the actual positives, how many you found.

Care more about recall when misses are costly: disease screening, fraud detection, safety issues. Care more about precision when false alarms are costly: spam filtering (flagging a legitimate email as spam is bad), pushing an intrusive intervention to users.

The two trade off against each other through the classification threshold: lower the threshold and recall rises while precision falls. In practice the threshold is set by the business costs of the two kinds of error.

**Short version:** Precision is the share of predicted positives that are correct; recall is the share of actual positives we catch. I prioritize recall when misses are costly, like fraud or disease screening, and precision when false alarms are costly. The threshold trades one for the other.

</details>

**🗣 Say it out loud:** What is the difference between ROC-AUC and PR-AUC? Which one do you use with heavy class imbalance?

<details>
<summary><b>Reference answer</b></summary>

The ROC curve plots TPR (recall) against FPR across thresholds; AUC means: pick a random positive and a random negative, and it is the probability the model scores the positive higher. It measures ranking ability and does not depend on the threshold or the class ratio.

The PR curve plots precision against recall. When positives are rare, the FPR denominator (total negatives) is huge, so even with many false alarms the FPR stays small and ROC-AUC looks overly optimistic; the PR curve directly reflects "how many flagged cases are real" and is more sensitive to performance on the positive class. So with severe imbalance, when you care about the positive class, look at PR-AUC.

**Short version:** ROC-AUC is the probability that a random positive is ranked above a random negative; it's insensitive to class balance. With rare positives it can look optimistic because the false-positive rate has a huge denominator, so PR-AUC, which focuses on the positive class, is more informative.

</details>

**🗣 Say it out loud:** How do you deal with class imbalance?

<details>
<summary><b>Reference answer</b></summary>

In order: (1) **change the metric**: not accuracy but PR-AUC, recall@precision, etc.; (2) split the data with **stratified sampling** so every split has the same share of positives; (3) **class weights** (`class_weight="balanced"`) or cost-sensitive learning; (4) **tune the threshold**: once the model outputs probabilities, choose the threshold from business costs instead of the default 0.5; (5) resampling (undersample negatives, oversample positives, SMOTE), only on the training set, keeping in mind that it breaks probability calibration; (6) collect more positives.

Don't open with SMOTE. Often (1)–(4) are enough.

**Short version:** First pick the right metric and use stratified splits. Then class weights and threshold tuning based on business costs. Resampling such as undersampling or SMOTE is a later option, applied to the training set only, and it distorts calibration.

</details>

## Checkpoint

You can explain: odds vs. log-odds, interpreting coefficients, log loss, the difference between L1 and L2 and why L1 is sparse, why to standardize, precision / recall / thresholds, ROC-AUC vs. PR-AUC.

On to **08b**.